# Task 2a: Corruption Classifier Optuna Search & Full Retraining

Hyperparameter tuning and full-schedule retraining for the Task 2a CNN classifier as specified in the assignment.

### Mathematical Formulation
$$p = f(\tilde{x}) = [p_{clean},\, p_{salt},\, p_{blur},\, p_{occlusion}], \quad \hat{k} = \arg\max_{k}\, p_k$$
$$\mathcal{L}_{CE} = -\sum_{k=1}^{4} y_k \log p_k$$

### Optuna Search Space Requirements
As required by the assignment, Optuna tunes:
1. **Learning Rate** ($\text{lr} \in [10^{-4}, 10^{-2}]$ log scale)
2. **Batch Size** ($B \in \{32, 64, 128\}$)
3. **Convolutional Channel Configuration** ($\text{base\_channels} \in \{16, 32, 64\}$)
4. **Dropout Rate** ($\text{dropout} \in [0.0, 0.5]$)
5. **Weight Decay** ($\text{weight\_decay} \in [10^{-5}, 10^{-3}]$ log scale)

### Retraining:
The winning configuration is retrained for `FULL_TRAIN_EPOCHS = 35` to produce `task2a_classifier_final_best.pt` for downstream hard routing and soft MoE initialization.

In [2]:
# 1. Config
N_TRIALS = 20
OPTUNA_EPOCHS = 15
FULL_TRAIN_EPOCHS = 35
STUDY_NAME = "task2a_classifier"
STORAGE_DB = "sqlite:///task2a_study.db"
SEED = 42

In [3]:
# 2. Imports & Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from constants import CONFIGS_ROOT
from src.device_utils import get_device, device_report
from src.datasets import PetDataset
from src.models_classifier import CorruptionClassifier
from src.train_loop import train_one_epoch_classifier, validate_classifier, set_seed
from src.checkpoint_utils import create_checkpoint_dict, save_checkpoint

set_seed(SEED)
device = get_device()

/home/zaifi/genai-assignment-1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[device] Using GPU via CUDA: NVIDIA GeForce RTX 5050 Laptop GPU


In [4]:
# 3. Data Loading
train_dataset = PetDataset(mode="train", corruption_mode="label", seed=SEED)
val_dataset = PetDataset(mode="val", corruption_mode="label", seed=SEED)

In [5]:
# 4. Optuna Objective
def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    base_channels = trial.suggest_categorical("base_channels", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = CorruptionClassifier(
        base_channels=base_channels,
        dropout=dropout,
        num_classes=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_acc = 0.0

    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_one_epoch_classifier(model, train_loader, optimizer, device)
        val_res = validate_classifier(model, val_loader, device)
        acc = val_res["val_acc"]
        if acc > best_acc:
            best_acc = acc
        trial.report(acc, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_acc

In [6]:
# 5. Run Optimization
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_DB,
    load_if_exists=True,
    direction="maximize",
    pruner=optuna.pruners.MedianPruner()
)
study.optimize(objective, n_trials=N_TRIALS)

print(f"Best Trial: #{study.best_trial.number} with Accuracy: {study.best_value:.2%}")
print(study.best_params)

[I 2026-10-01 21:09:11,207] A new study created in RDB with name: task2a_classifier
[I 2026-10-01 21:10:52,872] Trial 0 finished with value: 0.9836956521739131 and parameters: {'lr': 0.00010583456408916052, 'batch_size': 64, 'base_channels': 32, 'dropout': 0.3602611860461195, 'weight_decay': 0.0003168975243756387}. Best is trial 0 with value: 0.9836956521739131.
[I 2026-10-01 21:12:21,773] Trial 1 finished with value: 0.5271739130434783 and parameters: {'lr': 0.007519235308160889, 'batch_size': 64, 'base_channels': 32, 'dropout': 0.07978355823919536, 'weight_decay': 0.0004188789698862805}. Best is trial 0 with value: 0.9836956521739131.
[I 2026-10-01 21:13:52,528] Trial 2 finished with value: 0.266304347826087 and parameters: {'lr': 0.0047720917032701175, 'batch_size': 32, 'base_channels': 64, 'dropout': 0.094943407008998, 'weight_decay': 5.8224226681795313e-05}. Best is trial 0 with value: 0.9836956521739131.
[I 2026-10-01 21:15:23,697] Trial 3 finished with value: 0.9769021739130435 

Best Trial: #18 with Accuracy: 99.05%
{'lr': 0.0006152290064858183, 'batch_size': 64, 'base_channels': 16, 'dropout': 0.001969787061167662, 'weight_decay': 0.00017497671189681938}


In [8]:
# 6. Save Best Config & Retrain Winning Model on Full Schedule
best_config_path = CONFIGS_ROOT / "task2a_best_config.yaml"
with open(best_config_path, "w") as f:
    yaml.dump({
        "task": "task2a_corruption_classifier",
        "best_trial_number": int(study.best_trial.number),
        "best_accuracy": float(study.best_value),
        "params": study.best_params,
    }, f, indent=2)
print(f"[Config] Saved best classifier hyperparameters to {best_config_path}")

# Retrain winning configuration on full schedule
print(f"\n--- Retraining Winning Classifier ({FULL_TRAIN_EPOCHS} Epochs) ---")
p = study.best_params
train_loader = DataLoader(train_dataset, batch_size=p["batch_size"], shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=p["batch_size"], shuffle=False, num_workers=2)

winner_model = CorruptionClassifier(
    base_channels=p["base_channels"],
    dropout=p["dropout"],
    num_classes=4
).to(device)

optimizer = torch.optim.Adam(winner_model.parameters(), lr=p["lr"], weight_decay=p["weight_decay"])
scaler = torch.amp.GradScaler('cuda') if device.type == "cuda" else None
best_val_acc = 0.0
checkpoint_dir = RESEARCH_ROOT / "checkpoints"

for epoch in range(1, FULL_TRAIN_EPOCHS + 1):
    tr = train_one_epoch_classifier(winner_model, train_loader, optimizer, device, scaler=scaler)
    vl = validate_classifier(winner_model, val_loader, device)
    val_acc = vl["val_acc"]
    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
    ckpt = create_checkpoint_dict(
        winner_model, optimizer, epoch, vl["val_loss"],
        optuna_trial_number=int(study.best_trial.number),
        random_seed=SEED,
        extra_metadata={**p, "val_acc": val_acc}
    )
    save_checkpoint(
        checkpoint_dir=checkpoint_dir,
        prefix="task2a_classifier_final",
        ckpt_dict=ckpt,
        is_best=is_best,
        task_name="task2a",
        phase_name="final",
        device_name=device_report()["device_name"]
    )
    print(f"Epoch [{epoch:02d}/{FULL_TRAIN_EPOCHS:02d}] Train Acc: {tr['train_acc']:.2%} | Val Acc: {val_acc:.2%} {'*Best*' if is_best else ''}")

print(f"Final retrained classifier best accuracy: {best_val_acc:.2%}")

[Config] Saved best classifier hyperparameters to /home/zaifi/genai-assignment-1/research/configs/task2a_best_config.yaml

--- Retraining Winning Classifier (35 Epochs) ---


Epoch [01/35] Train Acc: 58.42% | Val Acc: 76.22% *Best*
Epoch [02/35] Train Acc: 80.74% | Val Acc: 84.92% *Best*
Epoch [03/35] Train Acc: 87.16% | Val Acc: 90.22% *Best*
Epoch [04/35] Train Acc: 90.62% | Val Acc: 83.15% 
Epoch [05/35] Train Acc: 89.95% | Val Acc: 93.75% *Best*
Epoch [06/35] Train Acc: 95.35% | Val Acc: 95.92% *Best*
Epoch [07/35] Train Acc: 95.55% | Val Acc: 93.34% 
Epoch [08/35] Train Acc: 95.89% | Val Acc: 96.33% *Best*
Epoch [09/35] Train Acc: 96.40% | Val Acc: 96.06% 
Epoch [10/35] Train Acc: 98.23% | Val Acc: 94.57% 
Epoch [11/35] Train Acc: 96.84% | Val Acc: 93.21% 
Epoch [12/35] Train Acc: 96.77% | Val Acc: 97.01% *Best*
Epoch [13/35] Train Acc: 98.51% | Val Acc: 97.96% *Best*
Epoch [14/35] Train Acc: 97.59% | Val Acc: 97.83% 
Epoch [15/35] Train Acc: 98.51% | Val Acc: 97.83% 
Epoch [16/35] Train Acc: 98.30% | Val Acc: 96.33% 
Epoch [17/35] Train Acc: 97.49% | Val Acc: 97.69% 
Epoch [18/35] Train Acc: 98.23% | Val Acc: 97.15% 
Epoch [19/35] Train Acc: 98.17% | 

## 7. Held-Out Test Images: Final Classifier

In [ ]:
# Evaluate the final full-schedule classifier over every deterministic test corruption.
from src.checkpoint_utils import load_checkpoint
from src.evaluation import PetBenchmarkDataset

final_test_model = CorruptionClassifier(base_channels=p["base_channels"], dropout=p["dropout"], num_classes=4).to(device)
final_test_path = RESEARCH_ROOT / "checkpoints" / "task2a_classifier_final_best.pt"
final_test_meta = load_checkpoint(final_test_path, final_test_model, device=device)
final_test_model.eval()
test_dataset = PetBenchmarkDataset()
class_names = ["clean", "salt", "blur", "occlusion"]
confusion = torch.zeros(4, 4, dtype=torch.int64)
preview = []
with torch.inference_mode():
    for idx in range(len(test_dataset)):
        corrupted, _, target, corruption, severity, filename = test_dataset[idx]
        logits = final_test_model(corrupted.unsqueeze(0).to(device))
        predicted = int(logits.argmax(1).item())
        confusion[target, predicted] += 1
        if len(preview) < 12:
            preview.append((corrupted, target, predicted, corruption, severity, filename))
accuracy = confusion.diag().sum().item() / confusion.sum().item()
print(f"Checkpoint epoch {final_test_meta.get('epoch')} | Test accuracy: {accuracy:.2%} | {confusion.sum().item()} cases")
print("Rows=true, columns=predicted; class order:", class_names)
print(confusion.numpy())
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, (image, target, predicted, corruption, severity, filename) in zip(axes.flat, preview):
    ax.imshow(image.permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(f"{filename}\ntrue={class_names[target]} pred={class_names[predicted]}")
    ax.axis("off")
for ax in axes.flat[len(preview):]: ax.axis("off")
plt.tight_layout()
